# dev-fingerprint — Exploration Notebook

**Detecting the AI Drift in Famous OSS Contributors**

This notebook walks through the full analysis pipeline:
1. Style signal extraction from commit patches
2. Quarterly aggregation into LLM scores
3. Change-point detection correlated with LLM milestones
4. Multi-developer comparison & visualizations

> Self-contained: runs entirely on synthetic data — no GitHub token needed.

In [1]:
import sys
sys.path.insert(0, "..")

from datetime import datetime, timezone
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from devfp.models import Commit, CommitFile, Language, LLMScore, LLM_MILESTONES
from devfp.analyzer.style import extract_metrics
from devfp.analyzer.llm_signals import aggregate_quarterly
from devfp.analyzer.temporal import detect_change_points, _nearest_milestone

print("✓ imports OK")

✓ imports OK


## 1. Style Signal Extraction

We feed raw commit patches through `CodeAnalyzer` and compare a *human-style* snippet vs an *LLM-style* snippet.

In [2]:
HUMAN_PATCH = """\
+def calc(x, y):
+    return x + y
+
+def proc(lst):
+    res = []
+    for i in lst:
+        if i > 0:
+            res.append(i * 2)
+    return res
+
+# TODO: fix edge case
+def get_user(id):
+    try:
+        return db.find(id)
+    except:
+        return None
"""

LLM_PATCH = """\
+def calculate_sum(first_value: int, second_value: int) -> int:
+    \"\"\"
+    Calculate the sum of two integers.
+
+    Args:
+        first_value: The first integer to add.
+        second_value: The second integer to add.
+
+    Returns:
+        The sum of first_value and second_value.
+    \"\"\"
+    return first_value + second_value
+
+def process_positive_items(items: list[int]) -> list[int]:
+    \"\"\"
+    Filter and double all positive items in the list.
+
+    Args:
+        items: A list of integers to process.
+
+    Returns:
+        A new list containing doubled values of all positive integers.
+    \"\"\"
+    result = []
+    for item in items:
+        if item > 0:
+            result.append(item * 2)
+    return result
+
+def get_user_by_id(user_id: int) -> Optional[User]:
+    \"\"\"Retrieve a user from the database by their ID.\"\"\"
+    try:
+        return database.find_by_id(user_id)
+    except DatabaseError as error:
+        logger.error(f"Failed to fetch user {user_id}: {error}")
+        return None
+    except Exception as unexpected_error:
+        logger.critical(f"Unexpected error retrieving user: {unexpected_error}")
+        raise
"""

def _make_commit(patch, sha="x", date=None):
    return Commit(
        sha=sha, author="dev",
        date=date or datetime(2023, 6, 1, tzinfo=timezone.utc),
        message="feat: update",
        files=[CommitFile(filename="f.py", language=Language.PYTHON,
                          additions=20, deletions=0, patch=patch)],
    )

m_human = extract_metrics(_make_commit(HUMAN_PATCH, sha="h"))
m_llm   = extract_metrics(_make_commit(LLM_PATCH,   sha="l"))

signals = ["comment_density", "docstring_coverage", "avg_identifier_length",
           "error_handling_density", "llm_score"]
labels  = ["Comment density", "Docstring coverage", "Identifier length",
           "Error handling", "LLM score"]

fig = go.Figure()
x = np.arange(len(signals))
for metrics, name, color in [
    (m_human, "Human style (pre-2022)", "#4a90d9"),
    (m_llm,   "LLM-assisted style",    "#e74c3c"),
]:
    values = [getattr(metrics, s) for s in signals]
    fig.add_trace(go.Bar(name=name, x=labels, y=values,
                         marker_color=color, opacity=0.85))

fig.update_layout(
    title="Style signals: human vs LLM-assisted commit",
    barmode="group", yaxis_title="Score",
    legend=dict(orientation="h", y=1.12),
    template="plotly_white", height=400,
)
fig.show()

## 2. Synthetic Developer Histories

We simulate 3 archetypes:
- **`torvalds`** — no drift, stays organic throughout
- **`gaearon`** — gradual drift starting Q3 2022 (possible LLM influence)
- **`yyx990803`** — sharp jump after ChatGPT launch (high LLM influence)

In [3]:
rng = np.random.default_rng(42)

def _make_timeline(author, scores_by_quarter):
    """Build a list[LLMScore] from a {(year,q): score} dict."""
    result = []
    for (year, q), score in sorted(scores_by_quarter.items()):
        start_month = (q - 1) * 3 + 1
        end_month   = q * 3
        end_day     = 30 if end_month in {4, 6, 9, 11} else 31
        result.append(LLMScore(
            author=author,
            period_start=datetime(year, start_month, 1, tzinfo=timezone.utc),
            period_end=datetime(year, end_month, end_day, tzinfo=timezone.utc),
            llm_score=round(float(score), 1),
            n_commits=rng.integers(8, 25),
        ))
    return result

def _noisy(base, n, noise=2.5):
    return base + rng.normal(0, noise, n)

quarters = [(y, q) for y in range(2020, 2025) for q in range(1, 5)]
quarters = [qtr for qtr in quarters if not (qtr[0] == 2024 and qtr[1] > 2)]
N = len(quarters)

# torvalds: flat low
torvalds_scores = dict(zip(quarters, _noisy(9, N, noise=1.5)))

# gaearon: gradual rise from Q3 2022
gaearon_base = list(_noisy(28, N, noise=2.5))
copilot_idx  = quarters.index((2022, 3))
for i in range(copilot_idx, N):
    drift = (i - copilot_idx) * 1.4
    gaearon_base[i] = min(28 + drift + rng.normal(0, 2), 62)
gaearon_scores = dict(zip(quarters, gaearon_base))

# yyx990803: sharp jump after ChatGPT (Q4 2022)
yyx_base  = list(_noisy(30, N, noise=2.5))
chatgpt_idx = quarters.index((2022, 4))
for i in range(chatgpt_idx, N):
    yyx_base[i] = min(58 + rng.normal(0, 3), 78)
yyx_scores = dict(zip(quarters, yyx_base))

timelines = {
    "torvalds":   _make_timeline("torvalds",   torvalds_scores),
    "gaearon":    _make_timeline("gaearon",    gaearon_scores),
    "yyx990803":  _make_timeline("yyx990803",  yyx_scores),
}

print(f"Generated {N} quarters × 3 developers = {N*3} data points")
print(f"Period: {quarters[0]} → {quarters[-1]}")

Generated 18 quarters × 3 developers = 54 data points
Period: (2020, 1) → (2024, 2)


## 3. LLM Score Timeline + Change-Point Detection

In [4]:
COLORS = {"torvalds": "#4a90d9", "gaearon": "#f39c12", "yyx990803": "#e74c3c"}
MILESTONE_COLORS = {"GitHub Copilot GA": "#27ae60",
                    "ChatGPT Launch": "#8e44ad", "GPT-4 Release": "#c0392b"}

fig = go.Figure()

# LLM milestone bands
for label, dt in LLM_MILESTONES.items():
    if label not in MILESTONE_COLORS:
        continue
    fig.add_vline(
        x=dt.timestamp() * 1000, line_dash="dot",
        line_color=MILESTONE_COLORS[label], line_width=1.5,
        annotation_text=label.replace("GitHub ", "").replace(" Launch", ""),
        annotation_position="top",
        annotation_font_size=10,
    )

# Score timelines + change-points
for author, scores in timelines.items():
    xs = [s.period_start for s in scores]
    ys = [s.llm_score    for s in scores]
    color = COLORS[author]

    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode="lines+markers", name=author,
        line=dict(color=color, width=2),
        marker=dict(size=5),
    ))

    cps = detect_change_points(author, scores, min_size=2, penalty=2.0, min_magnitude=8.0)
    for cp in cps:
        fig.add_trace(go.Scatter(
            x=[cp.date], y=[cp.value_after + 3],
            mode="markers", marker=dict(symbol="triangle-down", size=12, color=color),
            name=f"{author} change-point", showlegend=False,
            hovertext=f"Δ {cp.magnitude:+.1f} — {cp.nearest_llm_event or 'unknown'}",
        ))

# Verdict thresholds
fig.add_hrect(y0=40, y1=70,  fillcolor="#f39c12", opacity=0.05, line_width=0)
fig.add_hrect(y0=70, y1=100, fillcolor="#e74c3c", opacity=0.05, line_width=0)

fig.update_layout(
    title="LLM Score Timeline — 3 developer archetypes",
    xaxis_title="Quarter", yaxis_title="LLM Score (0–100)",
    yaxis=dict(range=[0, 85]),
    legend=dict(orientation="h", y=-0.15),
    template="plotly_white", height=500,
    annotations=[
        dict(x=1.01, y=55, xref="paper", yref="y", text="Possible", showarrow=False,
             font=dict(size=9, color="#f39c12"), textangle=-90),
        dict(x=1.01, y=80, xref="paper", yref="y", text="High", showarrow=False,
             font=dict(size=9, color="#e74c3c"), textangle=-90),
    ],
)
fig.show()

## 4. Multi-Developer Comparison

Baseline (pre-2022) vs post-LLM era scores across all 10 configured developers — using the full synthetic dataset.

In [5]:
# Synthetic summary for all 10 developers (matches FINDINGS.md)
DEVS = [
    ("torvalds",   "Linus Torvalds",   8.2,  9.1),
    ("antirez",    "antirez",          11.4, 12.8),
    ("dhh",        "DHH",              24.3, 26.1),
    ("tj",         "TJ Holowaychuk",   19.3, 21.1),
    ("gvanrossum", "Guido van Rossum", 22.1, 38.4),
    ("gaearon",    "Dan Abramov",      28.6, 47.3),
    ("Rich-Harris","Rich Harris",       31.2, 52.8),
    ("ry",         "Ryan Dahl",        27.8, 55.2),
    ("sindresorhus","Sindre Sorhus",   35.7, 61.4),
    ("yyx990803",  "Evan You",         29.4, 58.1),
]

logins   = [d[0] for d in DEVS]
names    = [d[1] for d in DEVS]
baseline = [d[2] for d in DEVS]
post_llm = [d[3] for d in DEVS]
drift    = [p - b for b, p in zip(baseline, post_llm)]

order = sorted(range(len(drift)), key=lambda i: drift[i])
names_s    = [names[i]    for i in order]
baseline_s = [baseline[i] for i in order]
post_s     = [post_llm[i] for i in order]
drift_s    = [drift[i]    for i in order]

drift_colors = [
    "#e74c3c" if d > 20 else "#f39c12" if d > 10 else "#27ae60"
    for d in drift_s
]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Baseline vs Post-LLM Score", "Style Drift (Δ score)"),
    column_widths=[0.6, 0.4])

fig.add_trace(go.Bar(
    y=names_s, x=baseline_s, name="Baseline (pre-2022)",
    orientation="h", marker_color="#4a90d9", opacity=0.8), row=1, col=1)
fig.add_trace(go.Bar(
    y=names_s, x=post_s, name="Post-LLM (2022+)",
    orientation="h", marker_color="#e74c3c", opacity=0.8), row=1, col=1)

fig.add_trace(go.Bar(
    y=names_s, x=drift_s, name="Drift",
    orientation="h", marker_color=drift_colors, opacity=0.9,
    text=[f"+{d:.1f}" if d > 0 else f"{d:.1f}" for d in drift_s],
    textposition="outside"), row=1, col=2)

fig.update_xaxes(title_text="LLM Score", row=1, col=1)
fig.update_xaxes(title_text="Δ Score", range=[0, 35], row=1, col=2)
fig.update_layout(
    title="Developer Style Drift — pre vs post LLM era",
    barmode="overlay", template="plotly_white",
    height=450, legend=dict(orientation="h", y=1.1),
)
fig.show()

## 5. Radar Chart — Style Profile

A style fingerprint as a radar chart: 6 signals for a human-organic profile vs an LLM-assisted profile.

In [6]:
radar_signals  = ["Comment<br>density", "Docstring<br>coverage", "Identifier<br>verbosity",
                   "Error<br>handling", "Type hint<br>usage", "Commit<br>structure"]
organic_values = [0.08, 0.15, 0.42, 0.04, 0.04, 0.30]
llm_values     = [0.22, 0.68, 0.66, 0.09, 0.20, 0.75]

def _radar(values, name, color, dash="solid"):
    v = values + [values[0]]
    s = radar_signals + [radar_signals[0]]
    return go.Scatterpolar(
        r=v, theta=s, name=name, fill="toself",
        line=dict(color=color, width=2, dash=dash),
        fillcolor=color, opacity=0.15,
    )

fig = go.Figure([
    _radar(organic_values, "Organic style (pre-2022)", "#4a90d9"),
    _radar(llm_values,     "LLM-assisted style",       "#e74c3c", dash="dot"),
])
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 0.85])),
    title="Style fingerprint: organic vs LLM-assisted",
    legend=dict(orientation="h", y=-0.1),
    template="plotly_white", height=480,
)
fig.show()

## 6. Reproduce with Real Data

Replace the synthetic data above with actual GitHub commits:

```python
import os
from devfp.collector.github import GitHubCollector
from devfp.collector.cache import Cache

cache     = Cache(".cache")
collector = GitHubCollector(token=os.environ["GITHUB_TOKEN"], cache=cache)

commits = await collector.fetch_commits("gaearon", max_commits=400)
metrics = [extract_metrics(c) for c in commits if c.files]
scores  = aggregate_quarterly("gaearon", metrics)
cps     = detect_change_points("gaearon", scores)
```

See `devfp analyze gaearon` for the full CLI workflow.